In [ ]:
import json

import pandas as pd

from vdl_tools.shared_tools.project_config import get_paths
from vdl_tools.shared_tools.taxonomy_mapping.taxonomy_mapping import (
    redistribute_funding_fracs,
)
from vdl_tools.shared_tools.climate_landscape.add_taxonomy_mapping import (
    load_one_earth_taxonomy,
    remove_mapping_name_suffix_from_taxonomy_results,
)

from vdl_tools.shared_tools.climate_landscape import (
    funding_mapping_combination_utils as fmcu,
)

In [ ]:
import marimo as mo

In [ ]:
TAXONOMY_FILE = "../shared-data/data/taxonomies/oneearth/OE Solutions Terms 20250502_expanded_VDL.xlsx"

META_FILENAME = "../climate-landscape/data/results/cb_cd_li_meta.json"
# Undistributed ones so we distribute to whichever levl
TAXONOMY_MAPPING_RESULTS_FILE = "../grantham-neglectedness/data/results/cft_one_earth_taxonomy_mapping_results.json"

DISTRIBUTED_FUNDING_LEVEL = 3

CANDID_FUNDING_FILE = (
    "../shared-data/data/candid/2025_08_19/candid_orgs_cleaned.xlsx"
)
FUNDING_ROUND_FILE = "../shared-data/data/crunchbase/2025_09_01/organizations_funding_rounds.json"

In [ ]:
# taxonomy = load_one_earth_taxonomy(
#     TAXONOMY_FILE,
#     add_geo_engineering=False
# )

# taxonomy_mapping_results = pd.read_json(TAXONOMY_MAPPING_RESULTS_FILE)

# taxonomy_mapping_results = remove_mapping_name_suffix_from_taxonomy_results(taxonomy_mapping_results, "one_earth_category")

In [ ]:
taxonomy_mapping_results

<marimo-table data-initial-value='[]' data-label='null' data-data='"[{\"uid\":\"02716540-7479-4f60-a96b-9a3a340088ab\",\"Name\":\"Chestnut Carbon\",\"text_for\":\"Chestnut Carbon is a nature-based carbon removal developer focused on creating high-quality forestry projects in the United States that adhere to rigorous verification standards, specifically the Gold Standard. The organization aims to help corporations, landowners, and NGOs achieve their net-zero carbon goals by restoring marginal lands into biodiverse forest ecosystems. Their projects not only capture carbon but also enhance local biodiversity, improve air and water quality, and provide economic opportunities for surrounding communities.Founded in 2022, Chestnut Carbon employs a team with extensive experience in carbon markets, forestry, and corporate leadership. The organization utilizes proprietary technology to efficiently measure and monitor carbon sequestration, ensuring the integrity and durability of their projects. They are committed to responsible land management practices, certified by the Forest Stewardship Council, and focus on creating long-term partnerships with local stakeholders.Chestnut Carbon's mission includes restoring over 100,000 acres of forest by 2030 and removing 100 million tons of carbon from the atmosphere. Their afforestation projects are designed to be additional, meaning they would not occur without the carbon credit market, and they prioritize community engagement and ecological resilience throughout their operations.\",\"FundingFrac\":0.3333333333,\"cat_level\":2,\"category\":\"Forest Recovery\",\"level0\":\"Nature Conservation\",\"level1\":\"Ecosystem Restoration\",\"level2\":\"Forest Recovery\",\"level3\":null,\"one_earth\":\"Forest Recovery\",\"pct\":99.9050637916,\"rank\":0.0,\"reranked_relevancy\":true,\"sim\":0.5114232195,\"taxonomy_mapping_id\":\"02716540-7479-4f60-a96b-9a3a340088ab_Forest Recovery\"},{\"uid\":\"02716540-7479-4f60-a96b-9a3a340088ab\",\"Name\":\"Chestnut Carbon\",\"text_for\":\"Chestnut Carbon is a nature-based carbon removal developer focused on creating high-quality forestry projects in the United States that adhere to rigorous verification standards, specifically the Gold Standard. The organization aims to help corporations, landowners, and NGOs achieve their net-zero carbon goals by restoring marginal lands into biodiverse forest ecosystems. Their projects not only capture carbon but also enhance local biodiversity, improve air and water quality, and provide economic opportunities for surrounding communities.Founded in 2022, Chestnut Carbon employs a team with extensive experience in carbon markets, forestry, and corporate leadership. The organization utilizes proprietary technology to efficiently measure and monitor carbon sequestration, ensuring the integrity and durability of their projects. They are committed to responsible land management practices, certified by the Forest Stewardship Council, and focus on creating long-term partnerships with local stakeholders.Chestnut Carbon's mission includes restoring over 100,000 acres of forest by 2030 and removing 100 million tons of carbon from the atmosphere. Their afforestation projects are designed to be additional, meaning they would not occur without the carbon credit market, and they prioritize community engagement and ecological resilience throughout their operations.\",\"FundingFrac\":0.3333333333,\"cat_level\":3,\"category\":\"Reforestation Partnerships (Reforestation)\",\"level0\":\"Nature Conservation\",\"level1\":\"Ecosystem Restoration\",\"level2\":\"Reforestation\",\"level3\":\"Reforestation Partnerships (Reforestation)\",\"one_earth\":\"Reforestation Partnerships (Reforestation)\",\"pct\":99.8845642408,\"rank\":1.0,\"reranked_relevancy\":true,\"sim\":0.50422269,\"taxonomy_mapping_id\":\"02716540-7479-4f60-a96b-9a3a340088ab_Reforestation Partnerships (Reforestation)\"},{\"uid\":\"02716540-7479-4f60-a96b-9a3a340088ab\",\"Name\":\"Chestnut Carbon\"

In [ ]:
def load_files(
    taxonomy_path,
    taxonomy_mapping_results_path,
    meta_df_path,
    funding_round_path,
    candid_funding_path,
    add_geo_engineering=False,
):
    taxonomy = load_one_earth_taxonomy(
        taxonomy_path, add_geo_engineering=add_geo_engineering
    )
    taxonomy_mapping_results = pd.read_json(taxonomy_mapping_results_path)
    if add_geo_engineering:
        category_suffix = "one_earth"
    else:
        category_suffix = "one_earth_category"
    taxonomy_mapping_results = (
        remove_mapping_name_suffix_from_taxonomy_results(
            taxonomy_mapping_results, category_suffix
        )
    )
    meta_df = pd.read_json(meta_df_path)
    round_df = fmcu.load_cb_round_data(funding_round_path)
    candid_funding_raw = pd.read_excel(candid_funding_path)
    candid_funding_long = fmcu.reshape_candid_funding(
        candid_funding_raw, id_col="id"
    )
    combined_funding_df = fmcu.combine_funding_data(
        round_df, candid_funding_long
    )

    return (
        taxonomy,
        taxonomy_mapping_results,
        meta_df,
        round_df,
        candid_funding_long,
        combined_funding_df,
    )

In [ ]:
from vdl_tools.shared_tools.attention_index.attention_index import (
    AttentionIndexer,
)

In [ ]:
(
    taxonomy,
    taxonomy_mapping_results,
    meta_df,
    round_df,
    candid_funding_long,
    combined_funding_df,
) = load_files(
    taxonomy_path=TAXONOMY_FILE,
    taxonomy_mapping_results_path=TAXONOMY_MAPPING_RESULTS_FILE,
    meta_df_path=META_FILENAME,
    funding_round_path=FUNDING_ROUND_FILE,
    candid_funding_path=CANDID_FUNDING_FILE,
    add_geo_engineering=True,
)

/Users/zeintawil/.pyenv/versions/3.10.13/envs/vdl-env/lib/python3.10/site-packages/pandas/core/tools/datetimes.py:557: RuntimeWarning: invalid value encountered in cast
  arr, tz_parsed = tslib.array_with_unit_to_datetime(arg, unit, errors=errors)
/Users/zeintawil/.pyenv/versions/3.10.13/envs/vdl-env/lib/python3.10/site-packages/pandas/core/tools/datetimes.py:557: RuntimeWarning: invalid value encountered in cast
  arr, tz_parsed = tslib.array_with_unit_to_datetime(arg, unit, errors=errors)


In [ ]:
aier = AttentionIndexer(
    taxonomy=taxonomy,
    taxonomy_mapping_results=taxonomy_mapping_results,
    taxonomy_mapping_id_col="uid",
    meta_df=meta_df.copy(),
    round_df=round_df.copy(),
    candid_funding_long=candid_funding_long.copy(),
    combined_funding_df=combined_funding_df.copy(),
    additional_rooting_factor=3,
    min_year=2018,
    max_year=2025,
)

In [ ]:
attention_index = aier.calculate_attention_index(max_level=3)
attention_index

2025-10-24 11:38:32,933 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:117} - Redistributing funding fractions


2025-10-24 11:38:38,583 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:187} - Filtering funding by year 2018 to 2025
2025-10-24 11:38:38,587 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:194} - Filtering round by rounds to include
2025-10-24 11:38:38,589 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:390} - Calculating attention index at level 0


2025-10-24 11:38:38,604 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:390} - Calculating attention index at level 1


2025-10-24 11:38:38,629 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:390} - Calculating attention index at level 2


2025-10-24 11:38:38,683 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:390} - Calculating attention index at level 3


In [ ]:
import altair as alt

_attention_index = attention_index

_sort_order = attention_index.sort_values(
    "zero_max_geometric_mean_level_1", ascending=False
)["tax_map_level1"].unique()
# replace _df with your data source
_chart = (
    alt.Chart(
        _attention_index,
        height=800,
        width=1200,
    )
    .mark_circle()
    .encode(
        x=alt.X(
            field="min_max_scale_min_max_geometric_mean_level_3",
            type="quantitative",
        ),
        # x=alt.X(field='min_max_scale_min_max_geometric_mean_level_2', type='quantitative'),
        y=alt.Y(field="tax_map_level1", type="nominal", sort=_sort_order),
        color=alt.Color(field="tax_map_level1", type="nominal"),
        tooltip=[
            alt.Tooltip(field="tax_map_level1"),
            alt.Tooltip(field="tax_map_level2"),
            alt.Tooltip(field="tax_map_level3"),
            alt.Tooltip(field="min_max_scale_min_max_geometric_mean_level_3"),
        ],
    )
    .properties(
        # height=290,
        # width='container',
        config={"axis": {"grid": True}}
    )
)
_chart

In [ ]:
start_year = 2010

approach_results = {}
solution_results = {}

for start_year in range(2010, 2021):
    end_year = start_year + 3
    print(start_year)
    _aier = AttentionIndexer(
        taxonomy=taxonomy,
        taxonomy_mapping_results=taxonomy_mapping_results,
        taxonomy_mapping_id_col="uid",
        meta_df=meta_df.copy(),
        round_df=round_df.copy(),
        candid_funding_long=candid_funding_long.copy(),
        combined_funding_df=combined_funding_df.copy(),
        additional_rooting_factor=3,
        min_year=start_year,
        max_year=end_year,
        rounds_to_include=["pre_seed", "seed", "angel", "series_a"],
        distributed_funding_level=3,
    )

    _approach_attention_index = _aier.calculate_attention_index(max_level=3)
    _approach_attention_index["start_year"] = start_year
    _approach_attention_index["end_year"] = end_year

    approach_results[start_year] = _approach_attention_index

    _solution_attention_index = _approach_attention_index.drop_duplicates(
        subset=["tax_map_level0", "tax_map_level1", "tax_map_level2"]
    ).copy()
    solution_results[start_year] = _solution_attention_index

2010


2025-10-24 11:38:39,812 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:117} - Redistributing funding fractions


2025-10-24 11:38:45,196 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:187} - Filtering funding by year 2010 to 2013


2025-10-24 11:38:45,218 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:194} - Filtering round by rounds to include
2025-10-24 11:38:45,218 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:390} - Calculating attention index at level 0
2025-10-24 11:38:45,228 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:390} - Calculating attention index at level 1


2025-10-24 11:38:45,244 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:390} - Calculating attention index at level 2


...

2025-10-24 11:39:46,179 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:117} - Redistributing funding fractions


2025-10-24 11:39:51,692 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:187} - Filtering funding by year 2020 to 2023
2025-10-24 11:39:51,698 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:194} - Filtering round by rounds to include
2025-10-24 11:39:51,700 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:390} - Calculating attention index at level 0


2025-10-24 11:39:51,716 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:390} - Calculating attention index at level 1


2025-10-24 11:39:51,738 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:390} - Calculating attention index at level 2


2025-10-24 11:39:51,793 - INFO - {/Users/zeintawil/dev/vdl/vdl-tools/vdl_tools/shared_tools/attention_index/attention_index.py:390} - Calculating attention index at level 3


In [ ]:
total_approach_results = pd.concat(approach_results.values())
total_solution_results = pd.concat(solution_results.values())

approach_metric_cols = [
    x for x in total_solution_results.columns if x.endswith("level_3")
]
approach_metric_cols

approach_metric_cols.append("tax_map_level3")

total_solution_results = total_solution_results.drop(
    approach_metric_cols, axis=1
)

In [ ]:
total_approach_results.to_json(
    "/Users/zeintawil/Downloads/approach_results.json", orient="records"
)


total_solution_results.to_json(
    "/Users/zeintawil/Downloads/solutions_results.json", orient="records"
)

In [ ]:
energy_total_solution_results = total_solution_results[
    total_solution_results["tax_map_level0"] == "Energy Transition"
]

energy_total_solution_results = energy_total_solution_results[
    energy_total_solution_results["tax_map_level2"].apply(
        lambda x: ":" not in x and "No_" not in x
    )
]

In [ ]:
# replace _df with your data source
_chart = (
    alt.Chart(energy_total_solution_results)
    .mark_line()
    .encode(
        x=alt.X(field="end_year", type="nominal"),
        y=alt.Y(
            field="min_max_scale_min_max_geometric_mean_level_2",
            type="quantitative",
        ),
        color=alt.Color(field="tax_map_level2", type="nominal"),
        tooltip=[
            alt.Tooltip(field="end_year", format=",.0f"),
            alt.Tooltip(
                field="min_max_scale_min_max_geometric_mean_level_2",
                aggregate="mean",
                format=",.2f",
            ),
            alt.Tooltip(field="tax_map_level2"),
            alt.Tooltip(field="tax_map_level1"),
        ],
    )
    .properties(
        height=290,
        width="container",
        title="Attention over Time Energy Transition",
        config={"axis": {"grid": False}},
    )
)
mo.ui.altair_chart(_chart)
_chart

In [ ]:
# replace _df with your data source
_chart = (
    alt.Chart(
        energy_total_solution_results[
            energy_total_solution_results["tax_map_level1"]
            == "Renewable Power"
        ]
    )
    .mark_line()
    .encode(
        x=alt.X(field="end_year", type="nominal"),
        y=alt.Y(
            field="min_max_scale_min_max_geometric_mean_level_2",
            type="quantitative",
        ),
        color=alt.Color(field="tax_map_level2", type="nominal"),
        tooltip=[
            alt.Tooltip(field="end_year", format=",.0f"),
            alt.Tooltip(
                field="min_max_scale_min_max_geometric_mean_level_2",
                aggregate="mean",
                format=",.2f",
            ),
            alt.Tooltip(field="tax_map_level2"),
            alt.Tooltip(field="tax_map_level1"),
        ],
    )
    .properties(
        height=290,
        width="container",
        title="Attention over Time Renewable Power",
        config={"axis": {"grid": False}},
    )
)

mo.ui.altair_chart(_chart)
_chart

In [ ]:
# replace _df with your data source
_chart = (
    alt.Chart(
        energy_total_solution_results[
            (
                energy_total_solution_results["tax_map_level1"]
                == "Renewable Power"
            )
            & (energy_total_solution_results["start_year"].isin([2010, 2020]))
        ]
    )
    .mark_line()
    .encode(
        x=alt.X(field="end_year", type="nominal"),
        y=alt.Y(
            field="min_max_scale_min_max_geometric_mean_level_2",
            type="quantitative",
        ),
        color=alt.Color(field="tax_map_level2", type="nominal"),
        tooltip=[
            alt.Tooltip(field="end_year", format=",.0f"),
            alt.Tooltip(
                field="min_max_scale_min_max_geometric_mean_level_2",
                aggregate="mean",
                format=",.2f",
            ),
            alt.Tooltip(field="tax_map_level2"),
            alt.Tooltip(field="tax_map_level1"),
        ],
    )
    .properties(
        height=290,
        width="container",
        title="Attention Change Renewable Power",
        config={"axis": {"grid": False}},
    )
)
mo.ui.altair_chart(_chart)
_chart

In [ ]:
# replace _df with your data source
total_subpill_results = total_solution_results.drop_duplicates(
    subset=["tax_map_level0", "tax_map_level1", "start_year"]
)

total_subpill_results = total_subpill_results[
    total_subpill_results["tax_map_level1"].apply(
        lambda x: "No_" not in x and "Cross" not in x
    )
]

_chart = (
    alt.Chart(total_subpill_results)
    .mark_line()
    .encode(
        x=alt.X(field="end_year", type="nominal"),
        y=alt.Y(
            field="min_max_scale_min_max_geometric_mean_level_1",
            type="quantitative",
        ),
        color=alt.Color(field="tax_map_level1", type="nominal"),
        tooltip=[
            alt.Tooltip(field="end_year", format=",.0f"),
            alt.Tooltip(
                field="min_max_scale_min_max_geometric_mean_level_1",
                aggregate="mean",
                format=",.2f",
            ),
            alt.Tooltip(field="tax_map_level1"),
        ],
    )
    .properties(
        height=290,
        width="container",
        title="Attention over Time by Subpillars",
        config={"axis": {"grid": False}},
    )
)
mo.ui.altair_chart(_chart)

<marimo-vega data-initial-value='{}' data-label='null' data-spec='{"config":{"view":{"continuousWidth":300,"continuousHeight":300},"axis":{"grid":false}},"data":{"url":"./@file/39682-19470226-PW0ar7K8.csv","format":{"type":"csv"}},"mark":{"type":"line"},"encoding":{"color":{"field":"tax_map_level1","type":"nominal"},"tooltip":[{"field":"end_year","format":",.0f"},{"aggregate":"mean","field":"min_max_scale_min_max_geometric_mean_level_1","format":",.2f"},{"field":"tax_map_level1"}],"x":{"field":"end_year","type":"nominal"},"y":{"field":"min_max_scale_min_max_geometric_mean_level_1","type":"quantitative"}},"height":290,"title":"Attention over Time by Subpillars","width":"container","$schema":"https://vega.github.io/schema/vega-lite/v5.20.1.json"}' data-chart-selection='true' data-field-selection='true'>

In [ ]:
# replace _df with your data source


_chart = (
    alt.Chart(
        total_subpill_results[
            total_subpill_results["start_year"].isin([2010, 2020])
        ]
    )
    .mark_line()
    .encode(
        x=alt.X(field="end_year", type="nominal"),
        y=alt.Y(
            field="min_max_scale_min_max_geometric_mean_level_1",
            type="quantitative",
        ),
        color=alt.Color(field="tax_map_level1", type="nominal"),
        tooltip=[
            alt.Tooltip(field="end_year", format=",.0f"),
            alt.Tooltip(
                field="min_max_scale_min_max_geometric_mean_level_1",
                aggregate="mean",
                format=",.2f",
            ),
            alt.Tooltip(field="tax_map_level1"),
        ],
    )
    .properties(
        height=290,
        width="container",
        title="Attention Change Subpillars",
        config={"axis": {"grid": False}},
    )
)
mo.ui.altair_chart(_chart)
# _chart

<marimo-vega data-initial-value='{}' data-label='null' data-spec='{"config":{"view":{"continuousWidth":300,"continuousHeight":300},"axis":{"grid":false}},"data":{"url":"./@file/7229-19470226-QWMpm3OA.csv","format":{"type":"csv"}},"mark":{"type":"line"},"encoding":{"color":{"field":"tax_map_level1","type":"nominal"},"tooltip":[{"field":"end_year","format":",.0f"},{"aggregate":"mean","field":"min_max_scale_min_max_geometric_mean_level_1","format":",.2f"},{"field":"tax_map_level1"}],"x":{"field":"end_year","type":"nominal"},"y":{"field":"min_max_scale_min_max_geometric_mean_level_1","type":"quantitative"}},"height":290,"title":"Attention Change Subpillars","width":"container","$schema":"https://vega.github.io/schema/vega-lite/v5.20.1.json"}' data-chart-selection='true' data-field-selection='true'>